# 05 · Tree-based models — RQ2: linear versus tree ensembles

> **RQ2.** How does the accuracy of threshold-crossing RUL prediction from DLinear and NLinear compare with tree-based
> models (Decision Tree, Random Forest, AdaBoost, Gradient Boosting, XGBoost, LightGBM), particularly when capacity
> falls below the range seen in training?

| | |
|---|---|
| **Inputs** | `processed/*`, `results/rq1_records.csv` (linear models, from 04) |
| **Outputs** | `results/rq2_*.csv`, `results/rq2_hypotheses.json`, `figures/05_*.png` |

### Design

* **Same windows, same roll-out.** Trees read the notebook-03 features of exactly the windows the linear models read,
  forecast the same *H* cycles, and reach end of life through the same roll-out engine.
* **Two target modes per tree.** *absolute* (targets in Ah — standard practice) and *delta* (level-relative features,
  targets = change from the last value — the tree analogue of NLinear). This separates model family from target
  normalisation.
* **Two training arms.**
  * *full range* — all training windows (the main comparison);
  * *range restricted* — training windows whose targets fall below 85% state of health are removed, so the
    end-of-life region is unseen in training (the extrapolation stress test; see 02 §6 for why it is needed).

### Pre-registered hypotheses

* **H2a** — each linear model achieves lower relative RUL error than each absolute-target tree model (full range).
* **H2b** — under range restriction, absolute-target trees degrade more than NLinear.
* **H2c** — under range restriction, delta-target trees degrade less than the same model with absolute targets.

All RQ2 tests form one Holm family. "Degradation" is the paired change in relative error from the full-range arm to
the restricted arm on the same unit, using seed 0 in both arms (the restricted arm runs one seed per model).

In [ ]:
# Make the repository importable, locally or on Google Colab.
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Once per session:
    #   !git clone https://github.com/<your-account>/battery-rul-linear.git /content/battery-rul-linear
    #   %pip install -q -r /content/battery-rul-linear/requirements.txt
    # Raw data on Drive: MyDrive/battery_rul/data/NASA_PCoE/*.mat and .../CALCE_CX2/CX2_*/
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/battery-rul-linear")
    os.environ.setdefault("BATTERY_RUL_DATA", "/content/drive/MyDrive/battery_rul/data")

ROOT = Path.cwd().resolve()
if not (ROOT / "battery_rul").is_dir():
    ROOT = ROOT.parent                          # launched from notebooks/
os.environ.setdefault("BATTERY_RUL_ROOT", str(ROOT))
sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from battery_rul import config
from battery_rul.dataset import load_cells, loco_folds
from battery_rul.evaluation import UNIT_COLS, seed_average, summary_table, wide
from battery_rul.experiment import ExperimentSettings, baseline_spec, linear_spec, rollout_examples, run_experiment, tree_spec
from battery_rul.plotting import (INK, INK_2, MUTED, color_of, error_distribution, rollout_panel, savefig,
                                  setup_style)
from battery_rul.report import HYPOTHESES, HypothesisResult, describe, finalize
from battery_rul.stats import friedman, pairwise, wilcoxon
from battery_rul.utils import load_table, markdown_table, record_run, save_json, save_table, timer

pd.set_option("display.width", 170)
setup_style()
record_run("05_modelling_tree_ml_models")
cells = load_cells()
REF = config.LABELS.primary_reference
folds = loco_folds(cells, REF)
METRIC = config.PRIMARY_METRIC
TREES_ABS = [f"{m} (absolute)" for m in config.TREE_MODELS]
TREES_DELTA = [f"{m} (delta)" for m in config.TREE_MODELS]
LINEARS = ["DLinear", "NLinear"]
print(f"{len(folds)} folds | tree seeds {config.TREE_SEEDS} | range-arm seeds {config.RANGE_ARM_SEEDS}")

## 1. Full-range arm: train and evaluate the tree models

In [ ]:
tree_specs = [tree_spec(m, mode) for mode in config.TREE_TARGET_MODES for m in config.TREE_MODELS]
with timer("RQ2 full-range run"):
    rec_trees, fits_trees, _ = run_experiment(cells, folds, tree_specs, config.TREE_SEEDS,
                                              ExperimentSettings(reference=REF, arm="full_range"))
save_table(rec_trees, "rq2_records_full_range")
save_table(fits_trees, "rq2_fits_full_range")

rec_lin = load_table("rq1_records")
rec_lin = rec_lin[rec_lin["model"].isin(LINEARS + ["LinearTrend"])]
records_full = pd.concat([rec_lin, rec_trees], ignore_index=True)
units_full = seed_average(records_full)
save_table(units_full, "rq2_units_full_range")
summary_full = summary_table(units_full)
save_table(summary_full, "rq2_summary_full_range")
print(markdown_table(summary_full))
summary_full

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 7))
error_distribution(units_full, METRIC, LINEARS + ["LinearTrend"] + TREES_ABS + TREES_DELTA, ax=ax,
                   title="Full-range arm: relative RUL error per unit (seeds averaged)")
savefig(fig, "05_error_distribution_full")
plt.show()

## 2. Full-range statistics and H2a

In [ ]:
all_models = LINEARS + ["LinearTrend"] + TREES_ABS + TREES_DELTA
table_full = wide(units_full, METRIC, all_models)
omnibus = friedman(table_full)
save_json(omnibus, "rq2_friedman_full_range")
print(f"Friedman over {omnibus['k_models']} models, {omnibus['n_units']} units: "
      f"chi2 = {omnibus['statistic']:.2f}, p = {omnibus['p_value']:.3g}")

pairs_h2a = [(lin, tree) for tree in TREES_ABS for lin in LINEARS]
pw_h2a = pairwise(table_full, pairs_h2a)          # diff = linear - tree; H2a predicts < 0
pw_desc = pairwise(table_full, [(lin, tree) for tree in TREES_DELTA for lin in LINEARS])
save_table(pw_h2a, "rq2_pairwise_h2a")
save_table(pw_desc, "rq2_pairwise_linear_vs_delta_trees")
pw_h2a.round(4)

## 3. Range-restricted arm

The same folds and origins, with the end-of-life region removed from every model's training windows. LinearTrend has
no training step, so its forecasts are unchanged — it is kept as the fixed reference.

In [ ]:
restricted_specs = ([linear_spec(m) for m in LINEARS] + [baseline_spec("LinearTrend")] + tree_specs)
settings_r = ExperimentSettings(reference=REF, arm="range_restricted",
                                range_restrict_soh=config.RANGE_RESTRICT_SOH)
with timer("RQ2 range-restricted run"):
    rec_r, fits_r, models_r = run_experiment(cells, folds, restricted_specs, config.RANGE_ARM_SEEDS, settings_r,
                                             keep_models=True)
save_table(rec_r, "rq2_records_range_restricted")
save_table(fits_r, "rq2_fits_range_restricted")
units_r = seed_average(rec_r)
summary_r = summary_table(units_r)
save_table(summary_r, "rq2_summary_range_restricted")
print(f"windows removed by restriction (median per fold): "
      f"{fits_r.groupby('fold')['n_removed_by_range'].first().median():.0f}")
summary_r

### Degradation from full range to restricted (seed 0 in both arms)

In [ ]:
seed0_full = records_full[records_full["seed"] == 0]
seed0_r = rec_r[rec_r["seed"] == 0]
key = ["model", *UNIT_COLS]
paired = seed0_full[key + [METRIC]].merge(seed0_r[key + [METRIC]], on=key, suffixes=("_full", "_restricted"))
paired["degradation"] = paired[f"{METRIC}_restricted"] - paired[f"{METRIC}_full"]
save_table(paired, "rq2_degradation")
deg = paired.groupby("model").agg(median_full=(f"{METRIC}_full", "median"),
                                  median_restricted=(f"{METRIC}_restricted", "median"),
                                  median_degradation=("degradation", "median"), n_units=("degradation", "size"))
deg = deg.sort_values("median_degradation")
deg

In [ ]:
fig, ax = plt.subplots(figsize=(8, 0.42 * len(deg) + 1.5))
for i, (m, r) in enumerate(deg.iterrows()):
    ax.plot([r["median_full"], r["median_restricted"]], [i, i], color=color_of(m), lw=2, alpha=0.7)
    ax.scatter(r["median_full"], i, color="white", edgecolor=color_of(m), s=45, zorder=3, lw=1.5)
    ax.scatter(r["median_restricted"], i, color=color_of(m), s=45, zorder=3)
ax.set_yticks(range(len(deg)), deg.index)
ax.set(xlabel="median relative RUL error", title="Full range (open) -> range restricted (filled)")
ax.grid(axis="y", visible=False)
savefig(fig, "05_range_restriction_effect")
plt.show()

### The training-range floor

An absolute-target tree predicts averages of training targets, so its forecast cannot fall below the lowest target it
was trained on. Under range restriction that floor sits above the EOL threshold, and the forecast can never cross it.
The table shows, per model, the share of units whose forecast stays at or above the training minimum all the way to the
true end of life.

In [ ]:
r0 = rec_r[rec_r["seed"] == 0].copy()
r0["stays_above_train_min"] = r0["forecast_min_Ah"] >= r0["train_target_min_Ah"] - 1e-6
floor = r0.groupby("model").agg(share_above_train_min=("stays_above_train_min", "mean"),
                                censor_rate=("censored", "mean")).sort_values("share_above_train_min")
save_table(floor.reset_index(), "rq2_training_floor")
floor

In [ ]:
target_cell = sorted({f.test for f in folds}, key=lambda n: (cells[n].dataset != "CALCE", n))[0]
show_models = ["NLinear", "DLinear"] + [m for m in ("XGBoost (absolute)", "XGBoost (delta)") if m in units_r["model"].unique()]
ex = rollout_examples(models_r, cells, [f for f in folds if f.test == target_cell], settings_r, show_models,
                      seed=config.RANGE_ARM_SEEDS[0])
fig, ax = plt.subplots(figsize=(9, 4.2))
rollout_panel(ex, cells[target_cell], 0.25, REF, ax=ax, legend_loc="lower left")
tmin = rec_r.loc[rec_r["cell"] == target_cell, "train_target_min_Ah"].iloc[0]
ax.axhline(tmin, color=MUTED, lw=1, ls=(0, (1, 2)))
ax.text(ax.get_xlim()[0], tmin, "  lowest training target (range restricted)", ha="left", va="bottom",
        fontsize=8, color=MUTED)
ax.set_title(f"{target_cell}, range-restricted training: roll-outs from 25% of life")
savefig(fig, "05_range_restricted_rollouts")
plt.show()

## 4. Hypotheses H2b and H2c, and the RQ2 verdicts

In [ ]:
dwide = paired.pivot_table(index=UNIT_COLS, columns="model", values="degradation").dropna()
results = []
for _, r in pw_h2a.iterrows():
    results.append(HypothesisResult(
        f"H2a[{r['model_a']} vs {r['model_b']}]", HYPOTHESES["H2a"], "Wilcoxon signed-rank (full range)",
        int(r["n_units"]), float(r["median_diff"]), "median(linear - tree) rel. error", float(r["p_value"]),
        bool(r["median_diff"] < 0)))
for tree in TREES_ABS:
    if {"NLinear", tree} <= set(dwide.columns):
        _, p = wilcoxon(dwide[tree], dwide["NLinear"])
        eff = float(np.median(dwide[tree] - dwide["NLinear"]))
        results.append(HypothesisResult(f"H2b[{tree}]", HYPOTHESES["H2b"],
                                        "Wilcoxon on paired degradation, tree vs NLinear", len(dwide), eff,
                                        "median(tree - NLinear) degradation", p, bool(eff > 0)))
for m in config.TREE_MODELS:
    a, d = f"{m} (absolute)", f"{m} (delta)"
    if {a, d} <= set(dwide.columns):
        _, p = wilcoxon(dwide[a], dwide[d])
        eff = float(np.median(dwide[a] - dwide[d]))
        results.append(HypothesisResult(f"H2c[{m}]", HYPOTHESES["H2c"],
                                        "Wilcoxon on paired degradation, absolute vs delta", len(dwide), eff,
                                        "median(absolute - delta) degradation", p, bool(eff > 0)))
rq2 = finalize(results)
rq2_df = pd.DataFrame([r.as_dict() for r in rq2])
save_table(rq2_df, "rq2_hypotheses")
save_json([r.as_dict() for r in rq2], "rq2_hypotheses")
rq2_df[["hypothesis", "n_units", "effect", "p_value", "p_holm", "verdict"]].round(4)

## 5. Summary

Generated from the results above.

In [ ]:
counts = rq2_df.assign(family=rq2_df["hypothesis"].str[:3]).groupby(["family", "verdict"]).size().unstack(fill_value=0)
print("Verdict counts per hypothesis family (Holm-corrected across all RQ2 tests):")
print(counts.to_string())
sf = summary_full.set_index("model")
best_tree = sf.loc[TREES_ABS + TREES_DELTA].sort_values("median_rel_err").index[0]
print(f"\nFull range: NLinear {sf.loc['NLinear', 'median_rel_err']:.3f}, DLinear {sf.loc['DLinear', 'median_rel_err']:.3f}, "
      f"best tree {best_tree} {sf.loc[best_tree, 'median_rel_err']:.3f}, LinearTrend {sf.loc['LinearTrend', 'median_rel_err']:.3f} "
      f"(median relative RUL error).")
abs_floor = floor.loc[[m for m in TREES_ABS if m in floor.index], "share_above_train_min"].mean()
lin_floor = floor.loc[[m for m in LINEARS if m in floor.index], "share_above_train_min"].mean()
print(f"Range restricted: absolute-target trees stay above their training minimum in {abs_floor:.0%} of units on "
      f"average, linear models in {lin_floor:.0%}.")
print("next: 06_window_length_explainable.ipynb (RQ3)")